这是**模拟退火算法 (Simulated Annealing, SA)** 的详细解析。

模拟退火算法由 Kirkpatrick 等人于 1983 年提出，其灵感源于固体退火原理。它是解决**组合优化问题**（如 TSP 旅商问题）和**复杂非凸函数优化**的一种经典随机搜索算法。

---

### 一、 数学原理与深度推导

**核心思想**：模拟物理退火过程，通过引入**随机性**（Metropolis 准则），赋予算法在搜索过程中以一定的概率接受“差解”的能力，从而能够**跳出局部最优解**，最终趋于全局最优。

#### 1. 物理背景与波尔兹曼分布
在热力学中，处于温度 $T$ 的物体，其分子状态能量为 $E$ 的概率服从**波尔兹曼分布**：
$$ P(E) = \frac{1}{Z(T)} \exp\left(-\frac{E}{k_B T}\right) $$
其中 $k_B$ 为波尔兹曼常数，$Z(T)$ 为归一化因子。
**数学含义**：温度越高，高能态出现的概率越大；温度越低，物体越倾向于停留在低能态（基态）。

#### 2. Metropolis 准则（数学脊梁）
假设当前解为 $x_{old}$，能量为 $E_{old} = f(x_{old})$；产生一个新解 $x_{new}$，其能量为 $E_{new} = f(x_{new})$。
能量差为：$\Delta E = E_{new} - E_{old}$。

算法接受新解的概率 $P$ 定义为：
$$ P = \begin{cases} 1, & \text{if } \Delta E < 0 \quad (\text{新解更好，直接接受}) \\ \exp\left(-\frac{\Delta E}{T}\right), & \text{if } \Delta E \ge 0 \quad (\text{新解更差，概率接受}) \end{cases} $$

**深度推导：为什么这个公式能跳出局部最优？**
*   当温度 $T$ 很大时，$\exp(-\Delta E/T)$ 接近 1。这意味着在算法早期（高温期），它几乎以 100% 的概率接受差解，这相当于在全空间进行**随机探索**。
*   随着 $T$ 逐渐降低，$\exp(-\Delta E/T)$ 变得极小。算法变得越来越“贪心”，只倾向于接受好解。
*   当 $T \to 0$ 时，算法演变为纯粹的**局部搜索（爬山法）**。

#### 3. 收敛性证明（马尔可夫链）
模拟退火在数学上被证明是**以概率 1 收敛到全局最优解**的。
**前提条件**：冷却进度足够慢（例如 $T_k = \frac{T_0}{\ln(1+k)}$）。在这种条件下，算法在每个温度下都能达到平稳分布（Stationary Distribution），该分布随温度降低会坍缩至全局最小值点。

---

### 二、 算法流程

1.  **初始化**：给定初温 $T_0$、终止温度 $T_{min}$、冷却速率 $\alpha$（如 0.95）、初始解 $x_0$。
2.  **产生新解**：在当前解 $x$ 的邻域内随机扰动产生 $x_{new}$。
3.  **计算代价**：$\Delta f = f(x_{new}) - f(x)$。
4.  **接受判定**：
    *   若 $\Delta f < 0$，接受新解 $x = x_{new}$。
    *   若 $\Delta f \ge 0$，产生 $[0, 1]$ 间随机数 $r$。若 $r < \exp(-\Delta f / T)$，接受新解；否则保持原解。
5.  **降温**：执行冷却计划 $T = \alpha \cdot T$。
6.  **终止**：若 $T < T_{min}$ 或达到最大迭代次数，停止并输出结果。

---

### 三、 适用场景
1.  **组合优化问题**：如 TSP 问题、装箱问题、排班问题。
2.  **非凸高维连续函数**：目标函数有成千上万个波峰波谷，梯度下降法一触即溃。
3.  **离散决策空间**：变量不是连续的，无法求导。

---

### 四 : Python 代码框架

我们将求解一个具有多个局部极小值的函数 $f(x) = x^2 + 10\sin(5x) + 7\cos(4x)$。

```python
import numpy as np
import matplotlib.pyplot as plt

# 1. 定义目标函数 (能量函数)
def objective_function(x):
    return x**2 + 10 * np.sin(5 * x) + 7 * np.cos(4 * x)

def simulated_annealing():
    # --- 1. 参数设置 ---
    T0 = 100.0        # 初始温度
    T_min = 1e-4      # 终止温度
    alpha = 0.98      # 降温系数 (冷却速率)
    max_iter = 100    # 每个温度下的迭代次数 (马尔可夫链长度)

    # 搜索空间 [-10, 10]
    x_current = np.random.uniform(-10, 10)
    f_current = objective_function(x_current)

    x_best = x_current
    f_best = f_current

    T = T0
    history = []

    # --- 2. 模拟退火过程 ---
    while T > T_min:
        for _ in range(max_iter):
            # (1) 产生新解 (扰动)
            # 扰动步长应随温度降低而减小，以提高局部搜索精度
            step = 0.5 * T / T0 + 0.1
            x_new = x_current + np.random.uniform(-step, step)
            x_new = np.clip(x_new, -10, 10) # 边界约束

            f_new = objective_function(x_new)
            delta_f = f_new - f_current

            # (2) Metropolis 准则判定
            if delta_f < 0: # 接受好解
                x_current, f_current = x_new, f_new
            else:
                # 以概率接受差解
                p = np.exp(-delta_f / T)
                if np.random.rand() < p:
                    x_current, f_current = x_new, f_new

            # (3) 更新全局最优记录
            if f_current < f_best:
                x_best, f_best = x_current, f_current

        history.append(f_best)
        T *= alpha # 降温

    return x_best, f_best, history

# ================= 运行示例 =================

if __name__ == "__main__":
    best_x, best_f, history = simulated_annealing()

    print("-" * 30)
    print(f"全局最优解 x: {best_x:.6f}")
    print(f"最小函数值 f(x): {best_f:.6f}")

    # 绘制函数图像和最优解
    x_range = np.linspace(-10, 10, 1000)
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.plot(x_range, objective_function(x_range), label='目标函数')
    plt.plot(best_x, best_f, 'ro', label='找到的最优解')
    plt.title("函数寻优展示")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history)
    plt.title("收敛过程 (Best Fitness)")
    plt.xlabel("降温周期")
    plt.ylabel("能量 (Energy)")

    plt.tight_layout()
    plt.show()
```

---

### 五、 深入数学推导：冷却进度与马尔可夫链

在论文中，描述 SA 的“专业度”主要体现在对这三个要素的论述：

1.  **初始温度 $T_0$ 的选取**：
    *   **数学要求**：$T_0$ 应足够大，使得初始接受率 $P \approx 1$。
    *   **推导建议**：可以写“通过随机采样计算初始邻域均值差 $\overline{\Delta f}$，利用 $T_0 = -\overline{\Delta f} / \ln(P_0)$ 设定初温，以确保初始搜索的全面性”。

2.  **冷却计划 (Cooling Schedule)**：
    *   **等比降温**：$T_{k+1} = \alpha T_k$（常用，快但易陷局部最优）。
    *   **对数降温**：$T_k = T_0 / \ln(1+k)$（慢但理论保证全局最优）。
    *   **论文加分**：讨论 $\alpha$ 对搜索效率的影响。$\alpha$ 越大，搜索越精细，但耗时增加。

3.  **内循环长度 $L$ (马尔可夫链长度)**：
    *   **意义**：在恒定温度下，算法需要进行 $L$ 次迭代以达到**热平衡状态**。只有达到热平衡，波尔兹曼分布的统计特性才成立。

---

### 六、 论文写作建议

1.  **参数对比表**：列出不同 $\alpha$ 和 $T_0$ 下的实验结果。这显示你对算法的**随机不确定性**有深刻认识。
2.  **对比爬山法**：在论文中加入一段对比测试，证明简单的贪心爬山法会死在局部波谷，而模拟退火能成功逃逸。
3.  **邻域结构定义**：对于 TSP 等组合优化问题，重点写你是如何“产生新解”的。
    *   **逆转 (Inverse)**：反转子序列。
    *   **交换 (Swap)**：随机交换两个城市。
    *   **插入 (Insert)**：移动一个城市位置。
    这三种操作的组合能显著提高搜索效率。

**下一类算法，你想听哪一个的深度数学推导？**